In [1]:
import pandas as pd
import numpy as np
import os
from IPython.display import display

# --- CONFIGURATION ---
input_csv_name = None
for f in os.listdir('.'): 
    # Pega o arquivo que termina em _processed.csv mas NÃO é o _features_processed.csv (output)
    if f.endswith('_processed.csv') and not f.endswith('_features_processed.csv'):
        input_csv_name = f
        break

if not input_csv_name:
    print("ERRO: Nenhum arquivo '*_processed.csv' de entrada encontrado.")
else:
    print(f"Processando arquivo: {input_csv_name}")

    # Lista FINAL de colunas desejadas (UNSW-NB15)
    target_columns = [
        'ts', 'id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate',
        'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit',
        'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean',
        'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm',
        'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login',
        'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports',
        'attack_cat', 'label'
    ]

    rename_map = {
        'duration': 'dur',
        'conn_state': 'state',
        'orig_pkts': 'spkts',
        'resp_pkts': 'dpkts',
        'orig_bytes': 'sbytes',
        'resp_bytes': 'dbytes',
        'http_trans_depth': 'trans_depth',
        'http_response_body_len': 'response_body_len'
        # sttl, dttl, sjit, etc já vêm com nomes certos do data-processing
    }

    # Valores padrão para preenchimento se coluna não existir
    default_values = {
        'sttl': 0, 'dttl': 0, 'sloss': 0, 'dloss': 0,
        'sinpkt': 0.0, 'dinpkt': 0.0, 'sjit': 0.0, 'djit': 0.0,
        'swin': 0, 'stcpb': 0, 'dtcpb': 0, 'dwin': 0,
        'tcprtt': 0.0, 'synack': 0.0, 'ackdat': 0.0,
        'trans_depth': 0, 'response_body_len': 0, 'is_ftp_login': 0,
        'ct_ftp_cmd': 0, 'ct_flw_http_mthd': 0
    }

    try:
        df = pd.read_csv(input_csv_name, low_memory=False)
        print(f"Lido: {len(df)} linhas.")

        # Renomeação inicial
        df.rename(columns=rename_map, inplace=True)


        # --- CONSOLIDAÇÃO trans_depth (HTTP + SMTP) ---
        # Seu schema final tem apenas 'trans_depth'. Para manter compatibilidade UNSW-like:
        # - Se existir http_trans_depth (renomeado para trans_depth via rename_map), mantém.
        # - Se existir smtp_trans_depth, usa para preencher trans_depth quando este estiver vazio/0.
        if 'smtp_trans_depth' in df.columns:
            df['smtp_trans_depth'] = pd.to_numeric(df['smtp_trans_depth'], errors='coerce')
            # garante trans_depth numérico
            df['trans_depth'] = pd.to_numeric(df.get('trans_depth', 0), errors='coerce').fillna(0)
            # preenche apenas onde trans_depth é 0 e smtp tem valor
            df.loc[(df['trans_depth'] == 0) & (df['smtp_trans_depth'].notna()), 'trans_depth'] = df.loc[(df['trans_depth'] == 0) & (df['smtp_trans_depth'].notna()), 'smtp_trans_depth']
            # opcional: se service estiver vazio e porta 25, marca smtp
            if 'service' in df.columns and 'id.resp_p' in df.columns:
                df.loc[(df['id.resp_p'] == 25) & (df['service'].isin(['-', '', 'nan'])), 'service'] = 'smtp'


        # Adiciona colunas faltantes com default
        for col in target_columns:
            if col not in df.columns:
                df[col] = default_values.get(col, 0)
        
        # --- PREPARAÇÃO PARA O LOOP ---
        # Ordenação crucial por tempo
        if 'ts' in df.columns:
            df_sorted = df.sort_values(by='ts').reset_index(drop=True)
        else:
            print("AVISO: Coluna 'ts' não encontrada. Assumindo ordem sequencial do arquivo.")
            df_sorted = df.copy()

        # Pré-processamento de colunas chave para evitar erros de tipo no loop
        # Convertendo tudo para string para comparação segura
        df_sorted['temp_src_ip'] = df_sorted['id.orig_h'].astype(str)
        df_sorted['temp_dst_ip'] = df_sorted['id.resp_h'].astype(str)
        df_sorted['temp_src_port'] = df_sorted['id.orig_p'].fillna(0).astype(int).astype(str)
        df_sorted['temp_dst_port'] = df_sorted['id.resp_p'].fillna(0).astype(int).astype(str)
        df_sorted['temp_service'] = df_sorted['service'].fillna('-').astype(str)
        
        # --- ENGENHARIA DE FEATURES AGREGADAS (ct_*) ---
        print("Calculando features de janela deslizante (Window=100)...")
        print("Isso pode levar alguns minutos dependendo do tamanho do arquivo...")
        
        window_size = 100
        num_rows = len(df_sorted)
        
        # Inicializa listas para armazenar resultados (mais rápido que alocar no DF a cada iteração)
        ct_srv_src_list = []
        ct_srv_dst_list = []
        ct_dst_ltm_list = []
        ct_src_ltm_list = []
        ct_src_dport_ltm_list = []
        ct_dst_sport_ltm_list = []
        ct_dst_src_ltm_list = []

        # Loop Principal
        for i in range(num_rows):
            # Define o início da janela (nunca menor que 0)
            start_index = max(0, i - window_size + 1)
            
            # Fatia a janela atual (inclusive a linha atual 'i')
            window = df_sorted.iloc[start_index : i + 1]
            current_row = df_sorted.iloc[i]
            
            # Extrai valores da linha atual para comparação
            cur_src_ip = current_row['temp_src_ip']
            cur_dst_ip = current_row['temp_dst_ip']
            cur_src_port = current_row['temp_src_port']
            cur_dst_port = current_row['temp_dst_port']
            cur_service = current_row['temp_service']

            # 1. ct_srv_src: No. de conexões com mesmo serviço e mesmo IP origem na janela
            c1 = window[(window['temp_service'] == cur_service) & (window['temp_src_ip'] == cur_src_ip)].shape[0]
            ct_srv_src_list.append(c1)

            # 2. ct_srv_dst: No. de conexões com mesmo serviço e mesmo IP destino na janela
            c2 = window[(window['temp_service'] == cur_service) & (window['temp_dst_ip'] == cur_dst_ip)].shape[0]
            ct_srv_dst_list.append(c2)

            # 3. ct_dst_ltm: No. de conexões com mesmo IP destino na janela
            c3 = window[window['temp_dst_ip'] == cur_dst_ip].shape[0]
            ct_dst_ltm_list.append(c3)

            # 4. ct_src_ltm: No. de conexões com mesmo IP origem na janela
            c4 = window[window['temp_src_ip'] == cur_src_ip].shape[0]
            ct_src_ltm_list.append(c4)

            # 5. ct_src_dport_ltm: No. de conexões com mesmo IP origem e mesma Porta destino
            c5 = window[(window['temp_src_ip'] == cur_src_ip) & (window['temp_dst_port'] == cur_dst_port)].shape[0]
            ct_src_dport_ltm_list.append(c5)

            # 6. ct_dst_sport_ltm: No. de conexões com mesmo IP destino e mesma Porta origem
            c6 = window[(window['temp_dst_ip'] == cur_dst_ip) & (window['temp_src_port'] == cur_src_port)].shape[0]
            ct_dst_sport_ltm_list.append(c6)

            # 7. ct_dst_src_ltm: No. de conexões com mesmo IP origem e mesmo IP destino
            c7 = window[(window['temp_src_ip'] == cur_src_ip) & (window['temp_dst_ip'] == cur_dst_ip)].shape[0]
            ct_dst_src_ltm_list.append(c7)

            if i % 1000 == 0 and i > 0:
                print(f"  Processado: {i}/{num_rows} linhas...", end='\r')
        
        print(f"\n  Loop concluído. Atribuindo colunas...")
        
        # Atribui listas de volta ao DataFrame
        df_sorted['ct_srv_src'] = ct_srv_src_list
        df_sorted['ct_srv_dst'] = ct_srv_dst_list
        df_sorted['ct_dst_ltm'] = ct_dst_ltm_list
        df_sorted['ct_src_ltm'] = ct_src_ltm_list
        df_sorted['ct_src_dport_ltm'] = ct_src_dport_ltm_list
        df_sorted['ct_dst_sport_ltm'] = ct_dst_sport_ltm_list
        df_sorted['ct_dst_src_ltm'] = ct_dst_src_ltm_list
        
        # --- FEATURE COMPLEXA (UNSW-like): ct_state_ttl (código por regra; NÃO é contagem) ---
        print("Calculando ct_state_ttl (UNSW-like: regra sobre state+sttl+dttl)...")

        if 'sttl' in df_sorted.columns and 'dttl' in df_sorted.columns and 'state' in df_sorted.columns:
            # TTL=0 no pipeline costuma significar "não observado" pelo Argus/merge.
            # Trate como missing para não distorcer distribuição.
            sttl = pd.to_numeric(df_sorted['sttl'], errors='coerce').replace(0, np.nan).fillna(0).astype(int)
            dttl = pd.to_numeric(df_sorted['dttl'], errors='coerce').replace(0, np.nan).fillna(0).astype(int)
            state = df_sorted['state'].fillna('').astype(str)

            ct = np.zeros(len(df_sorted), dtype=int)

            # Regras do UNSW (valores discretos)
            m1 = state.eq('FIN') & sttl.isin([62, 63, 254, 255]) & dttl.isin([252, 253])
            ct[m1] = 1

            m2 = state.eq('INT') & sttl.isin([0, 62, 254]) & dttl.eq(0)
            ct[m2] = 2

            m3 = state.eq('CON') & sttl.isin([62, 254]) & dttl.isin([60, 252, 253])
            ct[m3] = 3

            m4 = state.eq('ACC') & sttl.eq(254) & dttl.eq(252)
            ct[m4] = 4

            m5 = state.eq('CLO') & sttl.eq(254) & dttl.eq(252)
            ct[m5] = 5

            m7 = state.eq('REQ') & sttl.eq(254) & dttl.eq(0)
            ct[m7] = 7

            df_sorted['ct_state_ttl'] = ct
        else:
            df_sorted['ct_state_ttl'] = 0        # Seleção Final das 44 Colunas Oficiais
        df_final = df_sorted[target_columns]
        
        output_csv_name = input_csv_name.replace('_processed.csv', '_features_processed_timestamp.csv')
        df_final.to_csv(output_csv_name, index=False)
        print(f"Salvo com sucesso: {output_csv_name}")
        print("Amostra das features calculadas:")
        display(df_final[['ct_srv_src', 'ct_state_ttl', 'sttl', 'dttl', 'rate']].head())

    except Exception as e:
        print(f"Erro Crítico: {e}")
        import traceback
        traceback.print_exc()

Processando arquivo: analysis_processed.csv
Lido: 2818 linhas.
Calculando features de janela deslizante (Window=100)...
Isso pode levar alguns minutos dependendo do tamanho do arquivo...
  Processado: 2000/2818 linhas...
  Loop concluído. Atribuindo colunas...
Calculando ct_state_ttl (UNSW-like: regra sobre state+sttl+dttl)...
Salvo com sucesso: analysis_features_processed_timestamp.csv
Amostra das features calculadas:


,ct_srv_src,ct_state_ttl,sttl,dttl,rate
0,1,0,-1.0,-1.0,16.719084
1,2,0,-1.0,-1.0,30.462293
2,1,0,-1.0,-1.0,1.754011
3,3,0,-1.0,-1.0,9.544057
4,1,0,-1.0,-1.0,9.135362
